In [ ]:
import cptac
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn as sns
import os
import umap
import mofax as mofa
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mofapy2.run.entry_point import entry_point
os.environ['R_HOME'] = '/Library/Frameworks/R.framework/Resources'
from process_multiomics_for_mofa import read_csv_df,filter_genes, filter_genes_ppm, scale_df, make_long, print_non_numeric_values, read_csv_df

In [ ]:
#Read TFs
TF_df = pd.read_csv('data/TF_names_v_1.01.txt', header=None) #manually curated TFs proteins are obtained from https://humantfs.ccbr.utoronto.ca/
TF_df.shape
TF_list = TF_df[0].dropna().unique().tolist()
#print(TF_list)

In [ ]:
#read rna
file_path = 'data/TCGA_RPPA/TCGA_BLCA_UNC_RNAseq_RSEM_log2.cct'
index_col = 0
blca_rna = read_csv_df(file_path)
print(blca_rna.shape)
blca_rna = blca_rna.set_index('attrib_name')
blca_gene_tf_df = filter_genes(blca_rna, TF_list, axis=0)
blca_gene_tf_df.index = blca_gene_tf_df.index.astype(str) + '_rna' # rename after filtering
blca_gene_tf_df.shape

In [ ]:
# Step 2: Scale and plot
df = blca_gene_tf_df

# Step 3: (Optional) Standardize expression across samples
scaler = StandardScaler()
#blca_gene_scaled = pd.DataFrame(scaler.fit_transform(df.T).T, index=df.index, columns=df.columns)
blca_gene_scaled = scale_df(blca_gene_tf_df, axis =1)
print(blca_gene_scaled.shape)
# Step 4: Plot heatmap
sns.clustermap(blca_gene_scaled, cmap='vlag', figsize=(12, 12))
#plt.show()

In [ ]:
blca_gene_scaled = scale_df(blca_gene_tf_df, axis =1)
print(blca_gene_scaled.shape)
blca_tf_lrna = make_long(blca_gene_tf_df, "RNA")
print(blca_tf_lrna.head())

In [ ]:
#read and make protein df
#Read Tumor data
fp_t = 'data/TCGA_RPPA/TCGA_BLCA_RPPA_Gene__Firehose_RPPA.cct'
index_col = 0
blca_prot_t = read_csv_df(fp_t)
#blca_prot_t.head()        # Show first 5 rows
#Filter TFs
blca_prot_t = blca_prot_t.set_index('attrib_name')
blca_prot_t_tf_df = filter_genes(blca_prot_t, TF_list, axis=0)
blca_prot_t_tf_df.index = blca_prot_t_tf_df.index.astype(str) + '_protein'
#print(blca_prot_t_tf_df.shape)

blca_tf_protein_scaled = scale_df(blca_prot_t_tf_df, axis =1)
print(blca_tf_protein_scaled.shape)
blca_tf_lprotein = make_long(blca_tf_protein_scaled, "protein")
print(blca_tf_lprotein.head())

In [ ]:
print_non_numeric_values(blca_tf_lprotein, 'Value')
print_non_numeric_values(blca_tf_lrna, 'Value')

In [ ]:
#Merge all omic df to single input for MOFA
combined_blca = pd.concat([blca_tf_lrna, blca_tf_lprotein], axis=0)
print(combined_blca.shape)
combined_blca = combined_blca.rename(columns={'Patient_ID': 'sample', 'attrib_name': 'feature', 'Value':'value', 'View':'view'})
combined_blca['group'] = 'groupA'
combined_blca = combined_blca.drop_duplicates(subset=["group", "view", "feature", "sample"])
combined_blca = combined_blca.dropna(subset=['value'])
print(combined_blca.head())

In [ ]:
## Examine features in your dataset
## if you only have two datasets, please skip the last row
D = [len(combined_blca[combined_blca["view"] == "RNA"]["feature"].value_counts()), 
    len(combined_blca[combined_blca["view"] == "protein"]["feature"].value_counts())]

M = 2         # Number of views, please check your data
K = 10        # Number of factors you want to check
N = len(combined_blca[combined_blca["view"] == "protein"]["sample"].value_counts())    # Number of samples per group
G = 1         # Number of groups, please set as 1
print(D)
print(N)

In [ ]:
bent = entry_point()
bent.set_data_options(
    scale_groups = False, 
    scale_views = False
)
bent.set_data_df(combined_blca, likelihoods = ["gaussian", "gaussian"])

In [ ]:
print(combined_blca['value'].apply(type).value_counts())
invalid_rows = combined_blca[pd.to_numeric(combined_blca['value'], errors='coerce').isna()]
print(invalid_rows.head())

In [ ]:
bent.set_model_options(
    factors = 10, 
    spikeslab_weights = True, 
    ard_factors = True,
    ard_weights = True
)

In [ ]:
bent.set_train_options(
    iter = 1000, 
    convergence_mode = "medium", 
    startELBO = 1, 
    freqELBO = 1, 
    dropR2 = 0.001, 
    gpu_mode = False, 
    verbose = False, 
    seed = 42
)
bent.build()
bent.run()

In [ ]:
path='/Users/sathya/work/Data/TFactivity/MultiCancer/blca/'
combined_blca_out = os.path.join(path, "Combined_blca_view.hdf5")

bent.save(outfile=combined_blca_out)

In [ ]:
blca_out = mofa.mofa_model("/Users/sathya/work/Data/TFactivity/MultiCancer/blca/Combined_blca_view.hdf5")
blca_out

In [ ]:
metadata = pd.read_csv("data/TCGA_RPPA/TCGA_BLCA_MS_Clinical.csv") 
blca_metadata = metadata.T
print(blca_metadata)
samples = blca_out.get_samples()['sample']  # returns a dict: {group_name: [sample_ids]}
print(samples.shape)
# Step 2: UMAP embedding
factors_df = blca_out.get_factors()  # returns a dict with group names
embedding = umap.UMAP(n_neighbors=15, min_dist=0.1).fit_transform(factors_df)

umap_df = pd.DataFrame(embedding, columns=["UMAP1", "UMAP2"])
umap_df["Patient_ID"] = samples # list of sample IDs from MOFA
merged_df = umap_df.merge(blca_metadata, on="Patient_ID", how="left")
#print(merged_df.shape)

# Step 3: Plot
plt.figure(figsize=(8,8))
sns.scatterplot(data=merged_df, x="UMAP1", y="UMAP2", hue="pathologic_stage", palette="tab10", s=100)
plt.title("UMAP of MOFA Factors Colored by Clinical Stage")
plt.tight_layout()
plt.show()